# NCKH — V6: Guarded Selective Router

## Mục tiêu

V5 cho thấy hai tín hiệu trái ngược rất rõ:

- **Negative text** khá an toàn với positive nhưng suppression chưa đủ mạnh.
- **Negative text + negative visual exemplar** cực mạnh trên hard-negative khi route đúng, nhưng có thể phá positive khi Qwen route nhầm.

V6 không chạy lại CountGD++ và không đổi model. Thay vào đó, V6 học cách **quyết định khi nào được phép dùng exemplar**.

```text
Image + Target
      ↓
V4 box evidence
  ├─ boxes V4 KEEP
  └─ boxes V4 REJECT
      ↓
Qwen3-VL Router
      ↓
Target evidence + Confounder evidence
      ↓
Deterministic route

STRONG target evidence
    → BASELINE (protect positive)

WEAK target evidence
    → NEGATIVE TEXT or V4

NO target evidence + HIGH confounder evidence
    → exemplar candidate
    → second safety guard
        ABSENT     → NEGATIVE TEXT + EXEMPLAR
        PRESENT/?  → NEGATIVE TEXT

No reliable confounder
    → V4 fallback
```

## Điểm quan trọng

V6 **dùng lại kết quả đã cache của V5**:

- `baseline_count`
- `v4_filtered_count`
- `neg_text_count`
- `neg_text_exemplar_count`

Do đó V6 **không chạy lại CountGD++**, chỉ chạy Qwen3-VL để routing. Runtime phải ngắn hơn V4/V5 rất nhiều.

## Scientific note

V6 được thiết kế sau khi đã quan sát V4/V5 trên cùng benchmark. Vì vậy đây là **exploratory follow-up**, không phải untouched final test. Nếu V6 tốt, pipeline phải được freeze rồi xác nhận trên hard-negative set mới.


In [ ]:
# ============================================================
# CELL 1 — CONFIG
# ============================================================

from pathlib import Path
import random

SEED = 42
random.seed(SEED)

RUN_MODE = "FULL"   # "SMOKE" trước; ổn rồi đổi "FULL"

QWEN_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

# Evidence budget — chỉ dùng cho router, không phải CountGD inference.
MAX_KEEP_CROPS = 6
MAX_REJECT_CROPS = 6
PANEL_SIZE = 224
CROP_PADDING_FRAC = 0.30

# SMOKE luôn lấy đủ 3 nhóm.
SMOKE_HARD_OLD = 4
SMOKE_HARD_NEW = 4
SMOKE_POS = 5

SAVE_EVERY = 1
FAIL_FAST = False
RETRY_ERRORS = False

OUTPUT_DIR = Path("/kaggle/working/nckh_v6_guarded_router")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ROUTER_CSV = OUTPUT_DIR / "v6_router_predictions.csv"
GUARD_CSV = OUTPUT_DIR / "v6_guard_predictions.csv"
RESULTS_CSV = OUTPUT_DIR / "v6_results.csv"
METRICS_CSV = OUTPUT_DIR / "v6_metrics.csv"
METRICS_ORIGIN_CSV = OUTPUT_DIR / "v6_metrics_by_origin.csv"
BOOTSTRAP_JSON = OUTPUT_DIR / "v6_bootstrap.json"
FAILURES_CSV = OUTPUT_DIR / "v6_failure_cases.csv"
SUMMARY_JSON = OUTPUT_DIR / "v6_summary.json"

print("RUN_MODE:", RUN_MODE)
print("QWEN_MODEL_ID:", QWEN_MODEL_ID)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# ============================================================
# CELL 2 — INSTALL QWEN DEPENDENCIES ONLY
# V6 không cần CountGD++ repo/checkpoint.
# ============================================================

import sys, subprocess, importlib.metadata as im

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "transformers>=4.57.0,<5",
    "accelerate>=1.10.0",
    "qwen-vl-utils",
    "sentencepiece",
], check=True)

for pkg in ["torch", "transformers", "qwen-vl-utils"]:
    try:
        print(pkg, im.version(pkg))
    except Exception:
        pass


## Input cần Add vào Kaggle trước Cell 3

V6 cần đúng 3 nguồn:

1. **FSC-147 images** — dataset `hihihi5` như các phiên bản trước, có `images_384_VarV2`.
2. **V4 export** — `nckh_v4_qwen3_detection_verification_export.zip` hoặc dataset Kaggle đã giải nén chứa `v4_qwen3_predictions.csv`.
3. **V5 export** — `nckh_v5_native_negative_feedback_export.zip` hoặc dataset Kaggle đã giải nén chứa `v5_native_feedback_results.csv`.

V6 hỗ trợ cả **ZIP** lẫn **dataset đã giải nén** để tránh lỗi kiểu V5 trước đó.


In [ ]:
# ============================================================
# CELL 3 — FIND INPUTS ROBUSTLY (ZIP OR EXTRACTED DATASET)
# ============================================================

from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import json

SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]


def find_file(name):
    hits=[]
    for root in SEARCH_ROOTS:
        if root.exists():
            hits.extend([p for p in root.rglob(name) if p.is_file()])
    if not hits:
        return None
    return sorted(
        hits,
        key=lambda p: (0 if str(p).startswith("/kaggle/input") else 1, len(str(p)))
    )[0]


def find_image_dir():
    hits=[]
    for root in SEARCH_ROOTS:
        if root.exists():
            hits.extend([p for p in root.rglob("images_384_VarV2") if p.is_dir()])
    if not hits:
        return None
    return max(hits, key=lambda p: len(list(p.glob("*.jpg"))))


def resolve_export(zip_name, marker_name, extract_name):
    z = find_file(zip_name)
    marker = find_file(marker_name)

    if z is not None:
        out = Path("/kaggle/working") / extract_name
        out.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z, "r") as zf:
            zf.extractall(out)
        # Nếu ZIP có subfolder, tìm marker bên trong.
        markers = list(out.rglob(marker_name))
        if not markers:
            raise FileNotFoundError(f"ZIP {z} không chứa {marker_name}")
        root = markers[0].parent
        print(f"Resolved {extract_name} from ZIP:", z)
        return root

    if marker is not None:
        print(f"Resolved {extract_name} from extracted dataset:", marker.parent)
        return marker.parent

    raise FileNotFoundError(
        f"Không tìm thấy {zip_name} hoặc extracted marker {marker_name}"
    )


IMAGE_DIR = find_image_dir()
if IMAGE_DIR is None:
    raise FileNotFoundError("Không tìm thấy images_384_VarV2")

V4_ROOT = resolve_export(
    "nckh_v4_qwen3_detection_verification_export.zip",
    "v4_qwen3_predictions.csv",
    "v4_export_v6",
)

V5_ROOT = resolve_export(
    "nckh_v5_native_negative_feedback_export.zip",
    "v5_native_feedback_results.csv",
    "v5_export_v6",
)

V4_PRED = V4_ROOT / "v4_qwen3_predictions.csv"
V5_FEEDBACK = V5_ROOT / "v5_native_feedback_results.csv"
V5_DISCOVERY = V5_ROOT / "v5_confounder_discovery.csv"

for p in [V4_PRED, V5_FEEDBACK, V5_DISCOVERY]:
    if not p.exists():
        raise FileNotFoundError(p)

image_index = {p.name: str(p) for p in IMAGE_DIR.glob("*.jpg")}

pred = pd.read_csv(V4_PRED)
feedback = pd.read_csv(V5_FEEDBACK)
discovery_v5 = pd.read_csv(V5_DISCOVERY)

# Remap image path sang Kaggle input hiện tại.
feedback["image_path"] = feedback["image_name"].map(image_index)
pred["image_path"] = pred["image_name"].map(image_index)

if feedback["image_path"].isna().any():
    bad = feedback.loc[feedback["image_path"].isna(), "image_name"].tolist()
    raise FileNotFoundError(f"Thiếu ảnh FSC-147: {bad[:20]}")

required_feedback = {
    "group_id","pair_type","image_name","source_class","target","origin_batch","gt_count",
    "baseline_count","v4_filtered_count","decision","confounder",
    "neg_text_count","neg_text_exemplar_count"
}
missing = sorted(required_feedback - set(feedback.columns))
if missing:
    raise ValueError("V5 feedback thiếu cột: " + str(missing))

required_pred = {
    "crop_id","group_id","pair_type","image_name","target","box_json","score","vlm_keep","status"
}
missing = sorted(required_pred - set(pred.columns))
if missing:
    raise ValueError("V4 predictions thiếu cột: " + str(missing))

print("IMAGE_DIR:", IMAGE_DIR)
print("JPG count:", len(image_index))
print("V4 rows:", len(pred))
print("V5 groups:", len(feedback))
print("\nV5 pair types:")
print(feedback["pair_type"].value_counts())
print("\nHard origins:")
print(feedback.loc[feedback["pair_type"].eq("HARD_NEGATIVE"), "origin_batch"].value_counts())


In [ ]:
# ============================================================
# CELL 4 — BUILD RUN MANIFEST
# ============================================================

import pandas as pd

manifest = feedback.copy()

if RUN_MODE == "SMOKE":
    old = manifest[(manifest["pair_type"]=="HARD_NEGATIVE") & (manifest["origin_batch"]=="V3_OLD_56")]
    new = manifest[(manifest["pair_type"]=="HARD_NEGATIVE") & (manifest["origin_batch"]=="V4_NEW_REVIEWED")]
    pos = manifest[manifest["pair_type"]=="POSITIVE"]

    manifest_run = pd.concat([
        old.sample(min(SMOKE_HARD_OLD, len(old)), random_state=SEED),
        new.sample(min(SMOKE_HARD_NEW, len(new)), random_state=SEED+1),
        pos.sample(min(SMOKE_POS, len(pos)), random_state=SEED+2),
    ], ignore_index=True)
else:
    manifest_run = manifest.copy()

manifest_run = manifest_run.sort_values(
    ["pair_type","origin_batch","group_id"]
).reset_index(drop=True)

print("THIS RUN groups:", len(manifest_run))
print(manifest_run["pair_type"].value_counts())
print(manifest_run["origin_batch"].value_counts())
display(manifest_run[[
    "group_id","pair_type","image_name","target","origin_batch","gt_count",
    "baseline_count","v4_filtered_count","decision","confounder",
    "neg_text_count","neg_text_exemplar_count"
]].head(15))


In [ ]:
# ============================================================
# CELL 5 — PREPARE KEEP / REJECT EVIDENCE FROM V4
# ============================================================

import pandas as pd
import numpy as np

pred_ok = pred[pred["status"].astype(str).eq("ok")].copy()
pred_ok["vlm_keep"] = pd.to_numeric(pred_ok["vlm_keep"], errors="coerce").fillna(1).astype(int)
pred_ok["score"] = pd.to_numeric(pred_ok["score"], errors="coerce").fillna(0.0)

run_ids = set(manifest_run["group_id"].astype(str))
pred_run = pred_ok[pred_ok["group_id"].astype(str).isin(run_ids)].copy()

keep_boxes = pred_run[pred_run["vlm_keep"].eq(1)].copy()
reject_boxes = pred_run[pred_run["vlm_keep"].eq(0)].copy()

print("KEEP evidence boxes:", len(keep_boxes))
print("REJECT evidence boxes:", len(reject_boxes))
print("Groups with KEEP:", keep_boxes["group_id"].nunique(), "/", len(manifest_run))
print("Groups with REJECT:", reject_boxes["group_id"].nunique(), "/", len(manifest_run))

coverage = manifest_run[["group_id","pair_type","target"]].copy()
coverage = coverage.merge(
    keep_boxes.groupby("group_id").size().rename("keep_boxes_n"),
    on="group_id", how="left"
).merge(
    reject_boxes.groupby("group_id").size().rename("reject_boxes_n"),
    on="group_id", how="left"
)
coverage[["keep_boxes_n","reject_boxes_n"]] = coverage[["keep_boxes_n","reject_boxes_n"]].fillna(0).astype(int)
display(coverage.head(20))


In [ ]:
# ============================================================
# CELL 6 — CONTACT SHEETS + ROBUST JSON PARSER
# ============================================================

from PIL import Image, ImageDraw
import json, math, re


def parse_box(x):
    if isinstance(x, (list,tuple)):
        return [float(v) for v in x]
    return [float(v) for v in json.loads(str(x))]


def expanded_crop(image, box, padding_frac=CROP_PADDING_FRAC):
    x1,y1,x2,y2 = [float(v) for v in box]
    w=max(1.0,x2-x1); h=max(1.0,y2-y1)
    px=w*padding_frac; py=h*padding_frac
    ex1=max(0,int(math.floor(x1-px)))
    ey1=max(0,int(math.floor(y1-py)))
    ex2=min(image.width,int(math.ceil(x2+px)))
    ey2=min(image.height,int(math.ceil(y2+py)))
    return image.crop((ex1,ey1,ex2,ey2))


def make_sheet(image, rows, title, max_n):
    rows = list(rows)[:max_n]
    cols=3
    n=max(1,len(rows))
    nrows=int(math.ceil(n/cols))
    panel=PANEL_SIZE
    title_h=32
    canvas=Image.new("RGB", (cols*panel, title_h+nrows*panel), "white")
    draw=ImageDraw.Draw(canvas)
    draw.text((8,8), title, fill="black")

    if len(rows)==0:
        draw.text((8,title_h+10), "NO CANDIDATES", fill="black")
        return canvas

    for local_id,row in enumerate(rows):
        box=parse_box(row["box_json"])
        crop=expanded_crop(image,box)
        crop.thumbnail((panel-12,panel-30))
        x0=(local_id%cols)*panel
        y0=title_h+(local_id//cols)*panel
        px=x0+(panel-crop.width)//2
        py=y0+22+(panel-22-crop.height)//2
        canvas.paste(crop,(px,py))
        draw.rectangle([x0,y0,x0+panel-1,y0+panel-1], outline="black", width=2)
        draw.text((x0+6,y0+3),str(local_id),fill="black")
    return canvas


def extract_json_object(text):
    s=str(text).strip()
    # strip markdown fences
    s=re.sub(r"^```(?:json)?\s*", "", s, flags=re.I)
    s=re.sub(r"\s*```$", "", s)
    # first balanced-ish object via greedy boundaries
    start=s.find("{"); end=s.rfind("}")
    if start<0 or end<start:
        return None
    frag=s[start:end+1]
    try:
        return json.loads(frag)
    except Exception:
        # conservative cleanup of single quotes only as last resort
        try:
            return json.loads(frag.replace("'", '"'))
        except Exception:
            return None

print(extract_json_object('{"target_evidence":"NONE","confounder_evidence":"HIGH","confounder":"apples"}'))


In [ ]:
# ============================================================
# CELL 7 — LOAD QWEN3-VL-4B
# ============================================================

import torch, gc
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

if not torch.cuda.is_available():
    raise RuntimeError("V6 cần GPU Kaggle")

print("GPU:", torch.cuda.get_device_name(0))

qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
).eval()

print("Qwen loaded.")


In [ ]:
# ============================================================
# CELL 8 — ROUTER FUNCTION
# Output only evidence; deterministic routing is done in Python.
# ============================================================

import torch

@torch.inference_mode()
def qwen_route_evidence(image, target, keep_sheet, reject_sheet):
    prompt=f"""
You are a safety router for an object-counting system.

Target category: {target}

Image 1: full scene.
Image 2: candidate detections that a previous verifier KEPT as the target.
Image 3: candidate detections that a previous verifier REJECTED as not the target.

Your task is NOT to count.
Judge only visual evidence.

Return these fields:
1. target_evidence:
   - STRONG: at least one clear candidate/object is truly the target.
   - WEAK: target is plausible but not visually secure.
   - NONE: no credible target evidence is visible.
2. confounder_evidence:
   - HIGH: rejected crops are dominated by one coherent non-target category.
   - MEDIUM: a likely confounder exists but evidence is mixed.
   - LOW: rejected crops are mixed/unclear.
   - NONE: no usable rejected evidence.
3. confounder: short common noun phrase, or empty string.

Important:
- Similar-looking objects are not automatically the target.
- Do not infer from hidden labels.
- Do not use the source class; it is not provided.
- If the supposed confounder is a synonym/subtype/supertype/paraphrase of the target, set confounder_evidence to LOW.
- Be conservative about declaring target_evidence NONE.

Return exactly one JSON object and nothing else:
{{"target_evidence":"STRONG|WEAK|NONE","confounder_evidence":"HIGH|MEDIUM|LOW|NONE","confounder":"..."}}
""".strip()

    messages=[{
        "role":"user",
        "content":[
            {"type":"image","image":image},
            {"type":"image","image":keep_sheet},
            {"type":"image","image":reject_sheet},
            {"type":"text","text":prompt},
        ]
    }]

    text_prompt=qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        image_patch_size=16,
        return_video_kwargs=True,
        return_video_metadata=True,
    )

    video_metadatas=None
    if video_inputs is not None:
        video_inputs, video_metadatas = zip(*video_inputs)
        video_inputs=list(video_inputs); video_metadatas=list(video_metadatas)

    inputs=qwen_processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        video_metadata=video_metadatas,
        padding=True,
        return_tensors="pt",
        do_resize=False,
        **video_kwargs,
    ).to(next(qwen_model.parameters()).device)

    generated=qwen_model.generate(**inputs,max_new_tokens=120,do_sample=False)
    trimmed=[o[len(i):] for i,o in zip(inputs.input_ids,generated)]
    raw=qwen_processor.batch_decode(trimmed,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]
    return raw

print("Router function ready.")


In [ ]:
# ============================================================
# CELL 9 — RUN ROUTER WITH RESUME
# ============================================================

from PIL import Image
import pandas as pd
import time

existing=pd.read_csv(ROUTER_CSV) if ROUTER_CSV.exists() else pd.DataFrame()
records=existing.to_dict("records") if len(existing) else []

if len(existing):
    if RETRY_ERRORS:
        done=set(existing.loc[existing["status"].astype(str).eq("ok"),"group_id"].astype(str))
    else:
        done=set(existing["group_id"].astype(str))
else:
    done=set()

for idx,mrow in manifest_run.iterrows():
    gid=str(mrow["group_id"])
    if gid in done:
        continue

    k=keep_boxes[keep_boxes["group_id"].astype(str).eq(gid)].sort_values("score",ascending=False).head(MAX_KEEP_CROPS)
    r=reject_boxes[reject_boxes["group_id"].astype(str).eq(gid)].sort_values("score",ascending=False).head(MAX_REJECT_CROPS)

    image=Image.open(mrow["image_path"]).convert("RGB")
    keep_sheet=make_sheet(image,k.to_dict("records"),"V4 KEEP candidates",MAX_KEEP_CROPS)
    reject_sheet=make_sheet(image,r.to_dict("records"),"V4 REJECT candidates",MAX_REJECT_CROPS)

    t0=time.time(); raw=""; parsed=None; err=""; status="ok"
    try:
        raw=qwen_route_evidence(image,str(mrow["target"]),keep_sheet,reject_sheet)
        parsed=extract_json_object(raw)
        if parsed is None:
            status="parse_error"
            parsed={}
    except Exception as e:
        status="error"; err=repr(e); parsed={}
        if FAIL_FAST:
            raise

    te=str(parsed.get("target_evidence","WEAK")).upper()
    ce=str(parsed.get("confounder_evidence","LOW")).upper()
    conf=str(parsed.get("confounder","")).strip()

    if te not in {"STRONG","WEAK","NONE"}:
        te="WEAK"
    if ce not in {"HIGH","MEDIUM","LOW","NONE"}:
        ce="LOW"

    records.append({
        "group_id":gid,
        "pair_type":mrow["pair_type"],
        "image_name":mrow["image_name"],
        "target":mrow["target"],
        "origin_batch":mrow["origin_batch"],
        "target_evidence":te,
        "confounder_evidence":ce,
        "router_confounder":conf,
        "keep_boxes_shown":int(len(k)),
        "reject_boxes_shown":int(len(r)),
        "raw_output":raw,
        "latency_sec":round(time.time()-t0,3),
        "status":status,
        "error":err,
    })
    done.add(gid)
    pd.DataFrame(records).to_csv(ROUTER_CSV,index=False)
    print(f"[{idx+1}/{len(manifest_run)}] {gid} | target={mrow['target']} | target_ev={te} | conf_ev={ce} | conf={conf!r} | {status}")

router=pd.DataFrame(records)
router.to_csv(ROUTER_CSV,index=False)

print("\nTarget evidence by pair type:")
print(pd.crosstab(router["pair_type"],router["target_evidence"],normalize="index"))
print("\nConfounder evidence by pair type:")
print(pd.crosstab(router["pair_type"],router["confounder_evidence"],normalize="index"))


In [ ]:
# ============================================================
# CELL 10 — DETERMINISTIC PRE-GUARD ROUTER
# Frozen rule; Qwen does NOT choose the final method.
# ============================================================

import re


def norm_label(s):
    s=re.sub(r"[^a-z0-9 ]+"," ",str(s).lower())
    return " ".join(s.split())


def label_too_close(a,b):
    a=norm_label(a); b=norm_label(b)
    if not a or not b:
        return False
    if a==b or a in b or b in a:
        return True
    A=set(a.split()); B=set(b.split())
    if not A or not B:
        return False
    return len(A&B)/len(A|B) >= 0.75

joined=manifest_run.merge(
    router[["group_id","target_evidence","confounder_evidence","router_confounder","status"]],
    on="group_id",how="left",suffixes=("","_router")
)

routes=[]
for _,r in joined.iterrows():
    te=str(r["target_evidence"])
    ce=str(r["confounder_evidence"])
    v5_use=str(r["decision"]).upper()=="USE" and bool(str(r["confounder"]).strip())
    safe_conf=v5_use and not label_too_close(r["target"],r["confounder"])

    # Fail safe on router errors.
    if str(r["status_router"]) != "ok":
        route="V4"
    elif te=="STRONG":
        route="BASELINE"
    elif te=="NONE" and ce=="HIGH" and safe_conf:
        route="EXEMPLAR_CANDIDATE"
    elif te in {"NONE","WEAK"} and ce in {"HIGH","MEDIUM"} and safe_conf:
        route="NEG_TEXT"
    elif te=="NONE":
        route="V4"
    else:  # WEAK without reliable confounder
        route="V4"

    routes.append(route)

joined["pre_guard_route"]=routes
print(joined["pre_guard_route"].value_counts())
print("\nBy pair type:")
print(pd.crosstab(joined["pair_type"],joined["pre_guard_route"]))


In [ ]:
# ============================================================
# CELL 11 — SECOND SAFETY GUARD
# Only runs for EXEMPLAR_CANDIDATE.
# Goal: prevent catastrophic positive suppression.
# ============================================================

@torch.inference_mode()
def qwen_exemplar_guard(image,target,keep_sheet):
    prompt=f"""
Target category: {target}

Image 1 is the full scene.
Image 2 contains candidate detections previously KEPT as possible target instances.

This is a safety check before applying a very aggressive negative visual exemplar.
Question: Is there credible visual evidence that the target category itself is present?

Return exactly one JSON object:
{{"target_presence":"PRESENT|ABSENT|UNCERTAIN"}}

Rules:
- PRESENT if at least one object/candidate is clearly the target.
- ABSENT only if target is clearly not supported.
- UNCERTAIN if evidence is ambiguous.
- Be conservative: when in doubt, use UNCERTAIN rather than ABSENT.
""".strip()

    messages=[{"role":"user","content":[
        {"type":"image","image":image},
        {"type":"image","image":keep_sheet},
        {"type":"text","text":prompt},
    ]}]

    text_prompt=qwen_processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,image_patch_size=16,return_video_kwargs=True,return_video_metadata=True
    )
    video_metadatas=None
    if video_inputs is not None:
        video_inputs, video_metadatas=zip(*video_inputs)
        video_inputs=list(video_inputs); video_metadatas=list(video_metadatas)
    inputs=qwen_processor(
        text=[text_prompt],images=image_inputs,videos=video_inputs,video_metadata=video_metadatas,
        padding=True,return_tensors="pt",do_resize=False,**video_kwargs
    ).to(next(qwen_model.parameters()).device)
    generated=qwen_model.generate(**inputs,max_new_tokens=60,do_sample=False)
    trimmed=[o[len(i):] for i,o in zip(inputs.input_ids,generated)]
    raw=qwen_processor.batch_decode(trimmed,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]
    return raw

candidates=joined[joined["pre_guard_route"].eq("EXEMPLAR_CANDIDATE")].copy()
print("Need second guard:",len(candidates))

existing=pd.read_csv(GUARD_CSV) if GUARD_CSV.exists() else pd.DataFrame()
records=existing.to_dict("records") if len(existing) else []
done=set(existing["group_id"].astype(str)) if len(existing) else set()

for idx,r in candidates.iterrows():
    gid=str(r["group_id"])
    if gid in done:
        continue

    k=keep_boxes[keep_boxes["group_id"].astype(str).eq(gid)].sort_values("score",ascending=False).head(MAX_KEEP_CROPS)
    image=Image.open(r["image_path"]).convert("RGB")
    keep_sheet=make_sheet(image,k.to_dict("records"),"V4 KEEP candidates",MAX_KEEP_CROPS)

    t0=time.time(); raw=""; parsed=None; err=""; status="ok"
    try:
        raw=qwen_exemplar_guard(image,str(r["target"]),keep_sheet)
        parsed=extract_json_object(raw)
        if parsed is None:
            status="parse_error"; parsed={}
    except Exception as e:
        status="error"; err=repr(e); parsed={}
        if FAIL_FAST:
            raise

    presence=str(parsed.get("target_presence","UNCERTAIN")).upper()
    if presence not in {"PRESENT","ABSENT","UNCERTAIN"}:
        presence="UNCERTAIN"

    records.append({
        "group_id":gid,
        "target":r["target"],
        "target_presence":presence,
        "raw_output":raw,
        "latency_sec":round(time.time()-t0,3),
        "status":status,
        "error":err,
    })
    done.add(gid)
    pd.DataFrame(records).to_csv(GUARD_CSV,index=False)
    print(f"GUARD {gid} | {presence} | {status}")

guard=pd.DataFrame(records)
if len(guard):
    guard.to_csv(GUARD_CSV,index=False)
    print(guard["target_presence"].value_counts())
else:
    guard=pd.DataFrame(columns=["group_id","target_presence","status"])
    print("No exemplar candidates in this run.")


In [ ]:
# ============================================================
# CELL 12 — FINAL V6 ROUTE + COUNT FROM CACHED V5/V4 RESULTS
# ============================================================

result=joined.merge(
    guard[["group_id","target_presence","status"]].rename(columns={"status":"guard_status"}),
    on="group_id",how="left"
)

final_routes=[]
final_counts=[]

for _,r in result.iterrows():
    route=str(r["pre_guard_route"])

    if route=="EXEMPLAR_CANDIDATE":
        presence=str(r.get("target_presence","UNCERTAIN"))
        # Exemplar is allowed only when second guard says ABSENT.
        route = "NEG_TEXT_EXEMPLAR" if presence=="ABSENT" else "NEG_TEXT"

    if route=="BASELINE":
        count=int(r["baseline_count"])
    elif route=="V4":
        count=int(r["v4_filtered_count"])
    elif route=="NEG_TEXT":
        count=int(r["neg_text_count"])
    elif route=="NEG_TEXT_EXEMPLAR":
        count=int(r["neg_text_exemplar_count"])
    else:
        route="V4"
        count=int(r["v4_filtered_count"])

    final_routes.append(route)
    final_counts.append(count)

result["v6_route"]=final_routes
result["v6_count"]=final_counts

# Unguarded ablation: allow exemplar immediately.
def unguarded_count(r):
    route=str(r["pre_guard_route"])
    if route=="EXEMPLAR_CANDIDATE":
        return int(r["neg_text_exemplar_count"])
    if route=="BASELINE": return int(r["baseline_count"])
    if route=="NEG_TEXT": return int(r["neg_text_count"])
    return int(r["v4_filtered_count"])

result["v6_unguarded_count"]=[unguarded_count(r) for _,r in result.iterrows()]

# V5 exploratory hybrid reproduced for comparison.
result["v5_hybrid_count"]=np.where(
    result["decision"].astype(str).eq("USE"),
    result["neg_text_exemplar_count"],
    result["v4_filtered_count"],
).astype(int)

result.to_csv(RESULTS_CSV,index=False)

print("Final route distribution:")
print(result["v6_route"].value_counts())
print("\nBy pair type:")
print(pd.crosstab(result["pair_type"],result["v6_route"]))
display(result[[
    "group_id","pair_type","target","target_evidence","confounder_evidence",
    "pre_guard_route","target_presence","v6_route","baseline_count","v4_filtered_count",
    "neg_text_count","neg_text_exemplar_count","v6_count","gt_count"
]].head(20))


In [ ]:
# ============================================================
# CELL 13 — METRICS
# ============================================================

import numpy as np
import pandas as pd

methods={
    "COUNTGD_BASELINE":"baseline_count",
    "V4_BOX_FILTER":"v4_filtered_count",
    "V5_NEG_TEXT":"neg_text_count",
    "V5_NEG_TEXT_PLUS_EXEMPLAR":"neg_text_exemplar_count",
    "V5_HYBRID_EXPLORATORY":"v5_hybrid_count",
    "V6_ROUTER_UNGUARDED":"v6_unguarded_count",
    "V6_ROUTER_GUARDED":"v6_count",
}

rows=[]
for pair_type,g in result.groupby("pair_type"):
    y=g["gt_count"].to_numpy(float)
    for method,col in methods.items():
        p=g[col].to_numpy(float)
        row={
            "pair_type":pair_type,
            "method":method,
            "n":len(g),
            "mae":float(np.mean(np.abs(p-y))),
            "rmse":float(np.sqrt(np.mean((p-y)**2))),
            "mean_gt":float(np.mean(y)),
            "mean_pred":float(np.mean(p)),
        }
        if pair_type=="HARD_NEGATIVE":
            row["false_activation_rate"]=float(np.mean(p>0))
            row["zero_rate"]=float(np.mean(p==0))
        else:
            row["positive_survival_rate"]=float(np.mean(p>0))
        rows.append(row)

metrics=pd.DataFrame(rows)
metrics.to_csv(METRICS_CSV,index=False)
print("=== MAIN METRICS ===")
display(metrics.sort_values(["pair_type","mae"]))

origin_rows=[]
hard=result[result["pair_type"].eq("HARD_NEGATIVE")].copy()
for origin,g in hard.groupby("origin_batch"):
    for method,col in methods.items():
        p=g[col].to_numpy(float)
        origin_rows.append({
            "origin_batch":origin,
            "method":method,
            "n":len(g),
            "mae":float(np.mean(np.abs(p))),
            "fpr":float(np.mean(p>0)),
            "mean_pred":float(np.mean(p)),
            "zero_rate":float(np.mean(p==0)),
        })

metrics_origin=pd.DataFrame(origin_rows)
metrics_origin.to_csv(METRICS_ORIGIN_CSV,index=False)
print("\n=== HARD NEGATIVE BY ORIGIN ===")
display(metrics_origin.sort_values(["origin_batch","mae"]))


In [ ]:
# ============================================================
# CELL 14 — ROUTER DIAGNOSTICS
# This uses pair_type ONLY for evaluation, never as Qwen input.
# ============================================================

route_diag=pd.crosstab(result["pair_type"],result["v6_route"])
print("=== FINAL ROUTE BY TRUE PAIR TYPE ===")
display(route_diag)

# Dangerous routing metric: how many positives reach aggressive exemplar?
pos=result[result["pair_type"].eq("POSITIVE")]
hard=result[result["pair_type"].eq("HARD_NEGATIVE")]

print("Positive exemplar route rate:", float(np.mean(pos["v6_route"].eq("NEG_TEXT_EXEMPLAR"))))
print("Hard exemplar route rate:", float(np.mean(hard["v6_route"].eq("NEG_TEXT_EXEMPLAR"))))
print("Positive baseline protection rate:", float(np.mean(pos["v6_route"].eq("BASELINE"))))

# Guard effect specifically among exemplar candidates.
cand=result[result["pre_guard_route"].eq("EXEMPLAR_CANDIDATE")]
if len(cand):
    print("\nExemplar candidates by pair type:")
    print(cand["pair_type"].value_counts())
    print("\nGuard presence by pair type:")
    print(pd.crosstab(cand["pair_type"],cand["target_presence"]))


In [ ]:
# ============================================================
# CELL 15 — PAIRED BOOTSTRAP
# Negative point estimate => V6 has lower absolute error.
# ============================================================

import numpy as np, json

rng=np.random.default_rng(SEED)

def bootstrap_diff(a,b,n_boot=5000):
    a=np.asarray(a,float); b=np.asarray(b,float)
    assert len(a)==len(b)
    n=len(a)
    point=float(np.mean(a-b))
    vals=np.empty(n_boot,float)
    for i in range(n_boot):
        idx=rng.integers(0,n,size=n)
        vals[i]=np.mean(a[idx]-b[idx])
    lo,hi=np.quantile(vals,[0.025,0.975])
    return {"n":int(n),"point_estimate_A_minus_B":point,"ci95_low":float(lo),"ci95_high":float(hi)}

hard=result[result["pair_type"].eq("HARD_NEGATIVE")]
pos=result[result["pair_type"].eq("POSITIVE")]

boot={
    "interpretation":"Negative means V6 guarded has lower absolute error than comparator.",
    "hard_V6_minus_V4":bootstrap_diff(
        np.abs(hard["v6_count"]-hard["gt_count"]),
        np.abs(hard["v4_filtered_count"]-hard["gt_count"]),
    ),
    "hard_V6_minus_V5_hybrid":bootstrap_diff(
        np.abs(hard["v6_count"]-hard["gt_count"]),
        np.abs(hard["v5_hybrid_count"]-hard["gt_count"]),
    ),
    "positive_V6_minus_baseline":bootstrap_diff(
        np.abs(pos["v6_count"]-pos["gt_count"]),
        np.abs(pos["baseline_count"]-pos["gt_count"]),
    ),
    "positive_V6_minus_V4":bootstrap_diff(
        np.abs(pos["v6_count"]-pos["gt_count"]),
        np.abs(pos["v4_filtered_count"]-pos["gt_count"]),
    ),
}

with open(BOOTSTRAP_JSON,"w",encoding="utf-8") as f:
    json.dump(boot,f,indent=2)
print(json.dumps(boot,indent=2))


In [ ]:
# ============================================================
# CELL 16 — FAILURE CASES
# ============================================================

result["baseline_abs_error"]=(result["baseline_count"]-result["gt_count"]).abs()
result["v4_abs_error"]=(result["v4_filtered_count"]-result["gt_count"]).abs()
result["v6_abs_error"]=(result["v6_count"]-result["gt_count"]).abs()
result["v6_vs_v4_improvement"]=result["v4_abs_error"]-result["v6_abs_error"]
result["v6_vs_baseline_improvement"]=result["baseline_abs_error"]-result["v6_abs_error"]

cols=[
    "group_id","pair_type","image_name","source_class","target","origin_batch","gt_count",
    "target_evidence","confounder_evidence","router_confounder","decision","confounder",
    "pre_guard_route","target_presence","v6_route",
    "baseline_count","v4_filtered_count","neg_text_count","neg_text_exemplar_count","v6_count",
    "baseline_abs_error","v4_abs_error","v6_abs_error","v6_vs_v4_improvement","v6_vs_baseline_improvement"
]

failures=pd.concat([
    result.sort_values("v6_vs_v4_improvement").head(30),
    result.sort_values("v6_vs_v4_improvement",ascending=False).head(30),
],ignore_index=True).drop_duplicates("group_id")
failures[cols].to_csv(FAILURES_CSV,index=False)

print("=== MOST DEGRADED VS V4 ===")
display(result.sort_values("v6_vs_v4_improvement")[cols].head(15))
print("\n=== MOST IMPROVED VS V4 ===")
display(result.sort_values("v6_vs_v4_improvement",ascending=False)[cols].head(15))


In [ ]:
# ============================================================
# CELL 17 — SUMMARY + EXPORT ZIP
# ============================================================

import json, shutil
from pathlib import Path
from IPython.display import FileLink, display

summary={
    "phase":"V6_GUARDED_SELECTIVE_ROUTER",
    "run_mode":RUN_MODE,
    "qwen_model":QWEN_MODEL_ID,
    "primary_method":"V6_ROUTER_GUARDED",
    "design":{
        "countgd_rerun":False,
        "cached_sources":["V4 box predictions","V5 native feedback counts"],
        "router_inputs":["full image","V4 kept crops","V4 rejected crops","target text"],
        "aggressive_exemplar_rule":"only after router target_evidence=NONE + confounder_evidence=HIGH + V5 confounder USE + second guard=ABSENT",
        "positive_protection":"STRONG target evidence -> CountGD baseline",
        "uncertain_fallback":"V4 box filter or negative text, never force zero",
    },
    "n_groups":int(len(result)),
    "pair_type_counts":result["pair_type"].value_counts().to_dict(),
    "route_counts":result["v6_route"].value_counts().to_dict(),
    "route_by_pair_type":{
        str(k):{str(kk):int(vv) for kk,vv in v.items()}
        for k,v in result.groupby("pair_type")["v6_route"].value_counts().unstack(fill_value=0).to_dict(orient="index").items()
    },
    "metrics":metrics.to_dict("records"),
    "metrics_by_origin":metrics_origin.to_dict("records"),
    "bootstrap":boot,
    "scientific_notes":[
        "V6 is exploratory because V4/V5 results on this benchmark were already observed before designing the router.",
        "Ground-truth pair_type/source_class are retained only for evaluation and are never passed to Qwen router prompts.",
        "V6 does not rerun CountGD++; it selects among precomputed baseline, V4, negative-text, and negative-text+exemplar counts.",
        "A new held-out hard-negative benchmark is required before claiming final confirmatory improvement.",
    ],
}

with open(SUMMARY_JSON,"w",encoding="utf-8") as f:
    json.dump(summary,f,indent=2,ensure_ascii=False)

archive=shutil.make_archive(
    "/kaggle/working/nckh_v6_guarded_router_export",
    "zip",
    root_dir=str(OUTPUT_DIR),
)

print(json.dumps(summary,indent=2,ensure_ascii=False)[:18000])
print("\nCreated:",archive)
print("Size MB:",round(Path(archive).stat().st_size/1024**2,2))
display(FileLink(archive))


## Cách đọc V6

Ưu tiên 5 chỉ số:

1. `HARD_NEGATIVE / V6_ROUTER_GUARDED / mae` — thấp hơn V4 `14.96` là tín hiệu tốt.
2. `HARD_NEGATIVE / false_activation_rate` — thấp hơn V4 `23.42%` là tốt.
3. `POSITIVE / V6_ROUTER_GUARDED / mae` — mục tiêu là gần baseline `14.36`, tốt hơn V4 `16.71`.
4. `positive exemplar route rate` — lý tưởng gần `0%`; đây là lỗi nguy hiểm nhất của V5.
5. Bootstrap `hard_V6_minus_V4` và `positive_V6_minus_V4`.

### Thành công lý tưởng

```text
Hard MAE < 14.96
Hard FPR < 23.42%
Positive MAE <= ~16.7 và càng gần 14.36 càng tốt
Positive exemplar route ≈ 0%
```

Nếu V6 tốt trên benchmark hiện tại, bước sau **không được tiếp tục tune trên 111 pairs**. Hãy freeze pipeline và đánh giá trên một hard-negative set mới.
